In [3]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path(r"D:\Sami Data Set")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DOCUMENTATION_DIR = PROJECT_DIR / "documentation"

DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(RAW_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(f"No Parquet files found in: {RAW_DIR}")

print(f"Parquet files found: {len(parquet_files)}")

inventory_records = []

for file_path in parquet_files:
    try:
        parquet_file = pq.ParquetFile(file_path)

        columns = parquet_file.schema.names
        label_candidates = [
            column for column in columns
            if "label" in column.lower()
        ]

        inventory_records.append({
            "file_name": file_path.name,
            "size_mb": round(file_path.stat().st_size / (1024 ** 2), 2),
            "rows": parquet_file.metadata.num_rows,
            "columns": len(columns),
            "row_groups": parquet_file.metadata.num_row_groups,
            "label_candidates": ", ".join(label_candidates),
            "status": "Read successfully"
        })

    except Exception as error:
        inventory_records.append({
            "file_name": file_path.name,
            "size_mb": round(file_path.stat().st_size / (1024 ** 2), 2),
            "rows": None,
            "columns": None,
            "row_groups": None,
            "label_candidates": "",
            "status": f"Error: {error}"
        })

inventory = pd.DataFrame(inventory_records)

inventory_path = DOCUMENTATION_DIR / "dataset_inventory.csv"
inventory.to_csv(inventory_path, index=False)

display(inventory)

print("\nTotal files:", len(inventory))
print("Total rows:", inventory["rows"].sum())
print("Inventory saved to:", inventory_path)

Parquet files found: 10


,file_name,size_mb,rows,columns,row_groups,label_candidates,status
0,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,79.61,771587,78,1,Label,Read successfully
1,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,72.89,619346,78,1,Label,Read successfully
2,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,85.35,954846,78,1,Label,Read successfully
3,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,31.90,561396,78,1,Label,Read successfully
4,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,94.05,794812,78,1,Label,Read successfully
5,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,55.31,591873,78,1,Label,Read successfully
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,48.59,456873,78,1,Label,Read successfully
7,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,27.93,249170,78,1,Label,Read successfully
8,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,99.14,830224,78,1,Label,Read successfully
9,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,96.69,829405,78,1,Label,Read successfully



Total files: 10
Total rows: 6659532
Inventory saved to: D:\Sami Data Set\documentation\dataset_inventory.csv
